In [ ]:
from bs4 import BeautifulSoup,Tag
import requests

url = "https://sicstus.sics.se/sicstus/docs/latest4/html/sicstus.html/index.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')  # or 'lxml'

contents = soup.select_one('body > div > ul.no-bullet')
print(contents)

In [ ]:
def get_section(contents:Tag,section) -> Tag | None:
    for t in contents.descendants:
        if t.name == 'ul':
            possible_result = get_section(t,section)
            if possible_result:
                return possible_result
        elif t.name == 'li':
            if t.text.strip().split(' ')[0] == section:
                return t
        else:
            continue
    return None

In [ ]:
built_in_predicates = get_section(contents,"11.3")
built_in_predicates
number_builtins = len(list(built_in_predicates.children))
number_builtins

In [ ]:
teste = get_section(contents,"11.3.60")
teste

In [ ]:
for c in built_in_predicates.children:
    print(c)

In [ ]:
member_page_link = "https://sicstus.sics.se/sicstus/docs/latest4/html/sicstus.html/mpg_002dref_002dmember.html"
response = requests.get(member_page_link)
soup = BeautifulSoup(response.text, 'html.parser')  # or 'lxml'
soup

In [ ]:
def concat_ps(ps:list[Tag])->str:
    p_tags = [p for p in ps if p.name =='p']
    return "\n".join(p.text for p in p_tags)

class PredicateDocBuilder:
    def __init__(self):
        self.compat = None
        self.signature = None
        self.templates = []
        self.synopsis =None
        self.description = None
        self.arguments = None
        self.examples = None
        self.see_also = None
        self.exception = None
        self.backtracking =  None
        self.comments = None
        self.tips = None
        self.tags = []
        pass
    def handle_synopsis(self,contents):
        name, arity = self.signature
        # TODO if arity is list there will be more templates
        n_templates = 1
        contents = [c for c in contents if c.name is not None]
        if type(arity) is list:
            n_templates = len(arity)
        assert len(contents) > (n_templates), f"In predicate {name}/{arity} the synopsis is probably missing information"
        for i in range(n_templates):
            template_p = contents[i].text
            self.templates.append(template_p)
        self.synopsis = concat_ps(contents[n_templates:])

    def handle_description(self,contents):
        self.description =  concat_ps(contents)
    def handle_comments(self,contents):
        self.comments =  concat_ps(contents)

    def handle_tips(self,contents):
        self.tips =  concat_ps(contents)

    def handle_arguments(self,contents):
        for i in range(0,len(contents),2):
            term = contents[i]
            definition = contents[i+1]
        return 
    def handle_examples(self,contents):
        return 
    def handle_see_also(self,contents):
        self.see_also = concat_ps(contents)
    def handle_exception(self,contents):
        self.exceptions= concat_ps(contents)
    def handle_backtracking(self,contents):
        self.backtracking= concat_ps(contents)
    
    def handle_keywords(self,predicate_name):
        tags = predicate_name.select('i')
        for t in tags:
            self.tags.append(t.text)


    def handle_predicate_name(self,predicate_name):
        signature = predicate_name.text
        parts = signature.split('/')
        assert len(parts) == 2

        arity = parts[1]
        if parts[1][0] == '[':
            numbers = parts[1][1:-1].split(',')
            if all(n.isdigit() for n in numbers):
                arity = [int(n) for n in numbers]
            else:
                print(arity)
        elif parts[1].isdigit():
            arity = int(parts[1])
        self.signature = (parts[0],arity)

    def build(self,page):
        predicate_name = page.select("h4.subsection")
        assert len(predicate_name) == 1, "Found a page without predicate name"
        self.handle_predicate_name(predicate_name[0].select("code")[0])
        
        self.handle_keywords(predicate_name[0])

        subsections = page.select("h4.subheading")

        section_functions =  {
            'Synopsis': self.handle_synopsis,
            'Description': self.handle_description,
            'Arguments': self.handle_arguments,
            'Examples': self.handle_examples,
            'See Also': self.handle_see_also,
            'Exceptions': self.handle_exception,
            'Comments': self.handle_comments,
            'Tips': self.handle_tips,
            'Backtracking': self.handle_backtracking,
        }

        for section in subsections:
            name = section.text
            if name in section_functions:
                section_functions[name](self.section_contents(section))
            else:
                print("No registered function to handle:",name)
                raise TypeError

    def section_contents(self,tag:Tag)-> list[Tag]:
        result = []
        current_symbol = tag.next_sibling
        while current_symbol is not None and current_symbol.name != "h4":
            result.append(current_symbol)
            current_symbol = current_symbol.next_sibling
        return result
    def print(self):
        if self.compat != None:
            print(f"compat:{self.compat}")
        if self.signature != None:
            print(f"name:{self.signature}")
        if self.synopsis !=None:
            print(f"synopsis:{self.synopsis}")
        if self.description != None:
            print(f"description:{self.description}")
        if self.arguments != None:
            print(f"arguments:{self.arguments}")
        if self.examples != None:
            print(f"examples:{self.examples}")
        if self.see_also != None:
            print(f"see_also:{self.see_also}")
        if self.exception != None:
            print(f"exception:{self.exception}")
        if self.backtracking !=  None:
            print(f"backtracking:{self.backtracking}")
        if self.tags != []:
            print(f"tags:{self.tags}")


        

In [ ]:
predicate_builder = PredicateDocBuilder()
predicate_builder.build(soup)
predicate_builder.print()


In [ ]:
def visit_predicate(predicate_li:Tag):
    member_page_link = predicate_li.select('a')[0].get('href')
    base_link = "https://sicstus.sics.se/sicstus/docs/latest4/html/sicstus.html/"
    response = requests.get(base_link + member_page_link)
    if response.status_code != 200:
        raise Exception(response.status_code)
    soup = BeautifulSoup(response.text, 'html.parser')  # or 'lxml'
    predicate_builder = PredicateDocBuilder()
    predicate_builder.build(soup)
    name , _arity = predicate_builder.signature
    with open(f'builtins/{name}.html','w') as f:
        f.write(response.text)

    return predicate_builder

builtin_predicates = built_in_predicates.select("ul")[0].children

result = []
for predicate in builtin_predicates:
    if predicate.name is None:
        continue
    elif predicate.name == 'li':
        p = visit_predicate(predicate)
        result.append(p)
    elif predicate.name == 'ul':
        print(predicate)
    else:
        print(predicate)


In [ ]:
result[0].signature

In [ ]:
names = set()
for r in result:
    n = r.signature[0]
    if n in names:
        print(n)
    names.add(r.signature[0])
print(len(names))
print(len(result))